In [65]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.


In [66]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://kingcounty.gov/en/dept/dph/health-safety/disease-illness/foodborne-illness-outbreaks"

headers = {
    "User-Agent": "Mozilla/5.0"
}

detail_headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": BASE_URL
}

In [67]:
from urllib.parse import urljoin

# Download and parse the main outbreak webpage
response = requests.get(BASE_URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table")

outbreak_records = []

# Extract local King County outbreaks from the first three tables
for table in tables[:3]:
    for row in table.find_all("tr"):
        cells = row.find_all("td")

        # Skip header rows and rows without all four expected columns
        if len(cells) < 4:
            continue

        # Only keep rows containing a valid outbreak detail-page link
        link = cells[2].find("a", href=True)
        if link is None:
            continue

        outbreak_records.append({
            "date_posted": cells[0].get_text(" ", strip=True),
            "last_updated": cells[1].get_text(" ", strip=True),
            "food_business": cells[2].get_text(" ", strip=True),
            "suspected_organism": cells[3].get_text(" ", strip=True),
            "source_url": urljoin(BASE_URL, link["href"])
        })

outbreak_df = pd.DataFrame(outbreak_records)

In [68]:
# Visit each outbreak page and extract its additional information
all_details = []

for _, row in outbreak_df.iterrows():
    url = row["source_url"]

    try:
        page_response = requests.get(
            url, headers=detail_headers, timeout=30
        )
        page_response.raise_for_status()

        page_soup = BeautifulSoup(page_response.text, "html.parser")
        table = page_soup.find("table")

        details = {}

        if table:
            cells = table.find_all(["th", "td"])

            for i in range(1, len(cells) - 1, 2):
                label = cells[i].get_text(" ", strip=True)
                value = cells[i + 1].get_text(" ", strip=True)
                details[label] = value

        record = row.to_dict()
        record.update(details)
        all_details.append(record)

    except requests.RequestException as e:
        print(f"Failed to scrape {url}: {e}")

detailed_df = pd.DataFrame(all_details)

# Confirm every outbreak record has a detail-page URL
if outbreak_df["source_url"].isna().any():
    raise ValueError("Some outbreak records are missing URLs.")

In [69]:
# Standardize variations in column labels across outbreak pages
column_groups = {
    "cases": ["Cases"],
    "hospitalizations": ["Hospitalizations"],
    "deaths": ["Deaths"],
    "status": ["Status"],
    "location": ["Location"],
    "meal_date_range": [
        "Meal Date Range", "Meal Date", "Meal date",
        "Meal dates", "Meal date range"
    ],
    "illness_onset_date_range": [
        "Illness Onset Date Range", "Illness Onset Date",
        "Illness onset date range", "Illness onset date range:"
    ],
    "current_inspection_rating": [
        "Current Inspection Rating", "Current inspection rating"
    ],
    "date_suspected_products_were_purchased": [
        "Date suspected products were purchased:"
    ]
}

# Keep the main outbreak information
base_columns = [
    "date_posted", "last_updated", "food_business",
    "suspected_organism", "source_url"
]

cleaned_df = detailed_df[base_columns].copy()

# Combine differently named fields into standardized columns
for new_column, old_columns in column_groups.items():
    available_columns = [
        col for col in old_columns if col in detailed_df.columns
    ]

    cleaned_df[new_column] = (
        detailed_df[available_columns]
        .bfill(axis=1)
        .iloc[:, 0]
        if available_columns else None
    )

# Convert numerical fields and dates to appropriate data types
for column in ["cases", "hospitalizations", "deaths"]:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column], errors="coerce"
    )

# Convert dates with different formats into datetime objects
for column in ["date_posted", "last_updated"]:
    cleaned_df[column] = pd.to_datetime(
        cleaned_df[column],
        format="mixed",
        errors="coerce"
    )
# Standardize investigation status
status_mapping = {
    "investigation is complete": "Completed",
    "investigation is completed": "Completed",
    "investigation in completed": "Completed",
    "investigation is ongoing": "Ongoing"
}

cleaned_df["status"] = (
    cleaned_df["status"].str.strip().str.lower().map(status_mapping)
)

# Standardize inspection ratings
rating_mapping = {
    "good": "Good",
    "okay": "Okay",
    "needs to improve": "Needs to Improve",
    "excellent": "Excellent",
    "n/a": "N/A",
    "satisfactory": "Satisfactory"
}

cleaned_df["current_inspection_rating"] = (
    cleaned_df["current_inspection_rating"]
    .str.strip().str.lower().map(rating_mapping)
)

# Remove extra whitespace from text fields
text_columns = cleaned_df.select_dtypes(include=["object", "str"]).columns

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

In [70]:
# Export the cleaned database
cleaned_df.to_csv(
    "king_county_foodborne_outbreaks.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Database exported: {len(cleaned_df)} records")

Database exported: 30 records


In [110]:
import pandas as pd
import re
import string

# Standardize addresses
def clean_address(address):
    if pd.isna(address):
        return ""

    address = str(address).upper()
    address = address.translate(
        str.maketrans("", "", string.punctuation)
    )
    return " ".join(address.split())


# Standardize inspection addresses
inspections["address_match"] = (
    inspections["Address"].apply(clean_address)
)

In [111]:
# Standardize the full outbreak location
outbreaks["location_match"] = (
    outbreaks["location"].apply(clean_address)
)

In [112]:
# Get unique inspection addresses
inspection_addresses = (
    inspections[["Address", "City", "Business_ID", "address_match"]]
    .drop_duplicates()
)

matches = []

for _, outbreak in outbreaks.iterrows():

    location = outbreak["location_match"]

    for _, inspection in inspection_addresses.iterrows():

        address = inspection["address_match"]

        if address and address in location:

            matches.append({
                "food_business": outbreak["food_business"],
                "location": outbreak["location"],
                "inspection_address": inspection["Address"],
                "City": inspection["City"],
                "Business_ID": inspection["Business_ID"]
            })


# Create matching dataframe
address_matches = pd.DataFrame(matches).drop_duplicates()

print(
    "Restaurants matched:",
    address_matches["food_business"].nunique()
)

print(
    "Unique Business IDs:",
    address_matches["Business_ID"].nunique()
)

display(address_matches)

Restaurants matched: 29
Unique Business IDs: 40


,food_business,location,inspection_address,City,Business_ID
0,Cho Dang Tofu,"Cho Dang Tofu,1645 140th Ave NE A1-A, Bellevue...",1645 140TH AVE,BELLEVUE,PFE-PR-3135405
1,Cho Dang Tofu,"Cho Dang Tofu,1645 140th Ave NE A1-A, Bellevue...",1645 140TH AVE NE,BELLEVUE,PFE-PR-3130533
2,Cho Dang Tofu,"Cho Dang Tofu,1645 140th Ave NE A1-A, Bellevue...","1645 140TH AVE NE, A1-A",BELLEVUE,PFE-PR-3147640
3,Cho Dang Tofu,"Cho Dang Tofu,1645 140th Ave NE A1-A, Bellevue...",1645 140TH AVE NE,BELLEVUE,PFE-PR-3144249
4,Cho Dang Tofu,"Cho Dang Tofu,1645 140th Ave NE A1-A, Bellevue...",", 1",seattle,PFE-PR-502404
...,...,...,...,...,...
69,Coho Cafe,"6130 E Lake Sammamish Pkwy SE A, Issaquah, WA ...","6130 E LAKE SAMMAMISH PKWY SE, A",ISSAQUAH,PFE-PR-3134990
70,PCC Community Markets – West Seattle Co-op,PCC Community Markets – West Seattle Co-op 274...,2749 CALIFORNIA AVE SW,Seattle,PFE-PR-3149323
71,PCC Community Markets – West Seattle Co-op,PCC Community Markets – West Seattle Co-op 274...,", 1",seattle,PFE-PR-502404
72,Zobel Ethiopian Cuisine,"1219 E. Jefferson St., Seattle, WA 98122",", 1",seattle,PFE-PR-502404


In [107]:
matched_restaurants = address_matches["food_business"].unique()

unmatched_restaurants = outbreaks[
    ~outbreaks["food_business"].isin(matched_restaurants)
][["food_business", "location"]]

print("Unmatched restaurants:", len(unmatched_restaurants))

display(unmatched_restaurants)

Unmatched restaurants: 1


,food_business,location
22,Sammamish Cafe & Spirits,"Sammamish Cafe & Spirits, 22850 NE 8th St, Sam..."


In [113]:
# Match all inspection records using the identified Business IDs
matched_ids = address_matches["Business_ID"].unique()

matched_inspections = inspections[
    inspections["Business_ID"].isin(matched_ids)
].copy()

# Attach outbreak information
joined_df = matched_inspections.merge(
    address_matches[
        ["food_business", "Business_ID"]
    ].drop_duplicates(),
    on="Business_ID",
    how="left"
)

joined_df = joined_df.merge(
    outbreaks.drop(columns=["location_match"], errors="ignore"),
    on="food_business",
    how="left"
)

# Check results
print("Matched restaurants:", joined_df["food_business"].nunique())
print("Inspection records:", len(joined_df))
print("Unique inspections:",
      joined_df["Inspection_Serial_Num"].nunique())

Matched restaurants: 29
Inspection records: 509
Unique inspections: 242


In [114]:
joined_df.to_csv(
    "outbreak_restaurant_inspections.csv",
    index=False
)

print("Database exported!")

Database exported!


In [120]:
missing_data = pd.DataFrame({
    "Missing Count": joined_df.isnull().sum(),
    "Missing Percentage (%)": (
        joined_df.isnull().sum() / len(joined_df) * 100
    )
})

display(
    missing_data.sort_values(
        by="Missing Percentage (%)",
        ascending=False
    )
)

,Missing Count,Missing Percentage (%)
date_suspected_products_were_purchased,503,98.821218
Program Identifier,383,75.245580
Violation Description,126,24.754420
Violation Type,126,24.754420
Seating Range,76,14.931238
Parcel Number,36,7.072692
meal_date_range,6,1.178782
cases,0,0.000000
date_posted,0,0.000000
last_updated,0,0.000000


In [121]:

# Quantitative variables
quantitative_vars = [
    "Violation Points",
    "Inspection Score",
    "cases",
    "hospitalizations",
    "deaths"
]

display(joined_df[quantitative_vars].describe().T)


# Categorical variables
categorical_vars = [
    "Violation Type",
    "Inspection Result",
    "Inspection Type",
    "suspected_organism",
    "status"
]

for col in categorical_vars:
    print(f"\n{col}:")
    print(joined_df[col].value_counts(dropna=False))


,count,mean,std,min,25%,50%,75%,max
Violation Points,509.0,5.713163,6.035023,0.0,2.0,5.0,5.0,25.0
Inspection Score,509.0,24.165029,24.924221,0.0,2.0,17.0,35.0,86.0
cases,509.0,5.290766,6.480887,1.0,2.0,3.0,5.0,46.0
hospitalizations,509.0,0.369352,1.325523,0.0,0.0,0.0,0.0,10.0
deaths,509.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0



Violation Type:
Violation Type
RED     199
BLUE    184
NaN     126
Name: count, dtype: int64

Inspection Result:
Inspection Result
Unsatisfactory    325
Satisfactory      150
Complete           34
Name: count, dtype: int64

Inspection Type:
Inspection Type
Routine Inspection/Field Review    453
Educational Inspection              34
Return Inspection                   22
Name: count, dtype: int64

suspected_organism:
suspected_organism
Gastrointestinal illness                      263
Vibrio                                         56
Bacillus cereus or Clostridium perfringens     43
Norovirus                                      34
E. coli O121                                   28
Scombroid poisoning                            27
Shiga toxin-producing E. coli O157             17
Salmonella                                     17
Campylobacter                                  10
Shiga toxin-producing E. coli O121              8
Shiga toxin-producing E. coli O157:H7           6
Name: cou

In [122]:
quantitative_vars = [
    "Violation Points",
    "Inspection Score",
    "cases",
    "hospitalizations",
    "deaths"
]

quantitative_summary = joined_df[quantitative_vars].describe().T
display(quantitative_summary)

,count,mean,std,min,25%,50%,75%,max
Violation Points,509.0,5.713163,6.035023,0.0,2.0,5.0,5.0,25.0
Inspection Score,509.0,24.165029,24.924221,0.0,2.0,17.0,35.0,86.0
cases,509.0,5.290766,6.480887,1.0,2.0,3.0,5.0,46.0
hospitalizations,509.0,0.369352,1.325523,0.0,0.0,0.0,0.0,10.0
deaths,509.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
